# Data Enrichment

## Importing Libraries

In [6]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

#### Reading the concatinated data

In [2]:
OUTPUT_DIR = "../Data/citibike/"

In [10]:
citibike_df = pd.read_csv(f"{OUTPUT_DIR}/JC/JC2025.csv", 
                          parse_dates=['started_at','ended_at'])

### Missing Values

In [9]:
len(citibike_df)

48474

In [12]:
missing_values = citibike_df.isnull().sum().reset_index()

missing_values.columns = ['column_name', 'missing_count']

missing_values["missing_percentage"] = (missing_values["missing_count"] / len(citibike_df)) * 100

missing_values.sort_values(by="missing_percentage", ascending=False, inplace=True)

missing_values

,column_name,missing_count,missing_percentage
7,end_station_id,606,1.250155
10,end_lat,606,1.250155
11,end_lng,606,1.250155
6,end_station_name,352,0.726162
0,ride_id,0,0.000000
4,start_station_name,0,0.000000
3,ended_at,0,0.000000
2,started_at,0,0.000000
1,rideable_type,0,0.000000
8,start_lat,0,0.000000


In [13]:
citibike_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48474 entries, 0 to 48473
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ride_id             48474 non-null  str           
 1   rideable_type       48474 non-null  str           
 2   started_at          48474 non-null  datetime64[us]
 3   ended_at            48474 non-null  datetime64[us]
 4   start_station_name  48474 non-null  str           
 5   start_station_id    48474 non-null  str           
 6   end_station_name    48122 non-null  str           
 7   end_station_id      47868 non-null  str           
 8   start_lat           48474 non-null  float64       
 9   start_lng           48474 non-null  float64       
 10  end_lat             47868 non-null  float64       
 11  end_lng             47868 non-null  float64       
 12  member_casual       48474 non-null  str           
dtypes: datetime64[us](2), float64(4), str(7)
memory usage: 4.

#### Duration

In [15]:
citibike_df['ride_duration_min'] = (citibike_df["ended_at"] - citibike_df["started_at"]).dt.total_seconds()/60

In [16]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_min
0,FBA120ADE8D22E7D,electric_bike,2025-12-26 10:20:09.188,2025-12-26 10:40:13.508,Lafayette Park,JC078,Vesey St & Greenwich St,5216.07,40.713464,-74.062859,40.712547,-74.011131,casual,20.072000
1,DB8F52B8AE51DD28,electric_bike,2025-12-16 06:48:28.111,2025-12-16 06:56:10.969,Dr. Lena Edwards Park,JC117,Exchange Pl,JC116,40.713203,-74.058261,40.716366,-74.034344,member,7.714300
2,C16D1CA1746E065F,electric_bike,2025-12-20 12:50:55.361,2025-12-20 12:58:32.234,Dr. Lena Edwards Park,JC117,Exchange Pl,JC116,40.713203,-74.058261,40.716366,-74.034344,member,7.614550
3,7A613AF4BD6D75B9,electric_bike,2025-12-22 16:59:42.922,2025-12-22 17:04:29.327,JC Medical Center,JC110,Exchange Pl,JC116,40.715391,-74.049692,40.716366,-74.034344,casual,4.773417
4,2448F2B5B85E52B1,electric_bike,2025-12-17 18:06:09.352,2025-12-17 18:14:13.548,Baldwin at Montgomery,JC020,Exchange Pl,JC116,40.723659,-74.064194,40.716366,-74.034344,member,8.069933


In [17]:
citibike_df.shape

(48474, 14)

#### Filtering out anomalies

In [19]:
citibike_df = citibike_df[
    (citibike_df['ride_duration_min']>1) & (citibike_df['ride_duration_min']<24*60)
]

citibike_df.shape

(48417, 14)

In [20]:
missing_values = citibike_df.isnull().sum().reset_index()

missing_values.columns = ['column_name', 'missing_count']

missing_values["missing_percentage"] = (missing_values["missing_count"] / len(citibike_df)) * 100

missing_values.sort_values(by="missing_percentage", ascending=False, inplace=True)

missing_values

,column_name,missing_count,missing_percentage
10,end_lat,553,1.142161
7,end_station_id,553,1.142161
11,end_lng,553,1.142161
6,end_station_name,299,0.617552
0,ride_id,0,0.000000
1,rideable_type,0,0.000000
5,start_station_id,0,0.000000
4,start_station_name,0,0.000000
3,ended_at,0,0.000000
2,started_at,0,0.000000


#### Droping Missing Values


In [22]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'ride_duration_min'],
      dtype='str')

In [24]:
['ride_id', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       ]

['ride_id',
 'started_at',
 'ended_at',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng']

In [25]:
citibike_df = citibike_df.dropna(
    subset=['ride_id', 'started_at', 'ended_at','start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
        ]
)

In [26]:
missing_values = citibike_df.isnull().sum().reset_index()

missing_values.columns = ['column_name', 'missing_count']

missing_values["missing_percentage"] = (missing_values["missing_count"] / len(citibike_df)) * 100

missing_values.sort_values(by="missing_percentage", ascending=False, inplace=True)

missing_values

,column_name,missing_count,missing_percentage
0,ride_id,0,0.0
1,rideable_type,0,0.0
2,started_at,0,0.0
3,ended_at,0,0.0
4,start_station_name,0,0.0
5,start_station_id,0,0.0
6,end_station_name,0,0.0
7,end_station_id,0,0.0
8,start_lat,0,0.0
9,start_lng,0,0.0


#### Adding time granularity

In [28]:
citibike_df['date'] = citibike_df['started_at'].dt.date
citibike_df['month'] = citibike_df['started_at'].dt.to_period('M').astype(str)
citibike_df['month_name'] = citibike_df['started_at'].dt.month_name()
citibike_df['month_number'] = citibike_df['started_at'].dt.month
citibike_df['day_of_week'] = citibike_df['started_at'].dt.day_name()
citibike_df['hour'] = citibike_df['started_at'].dt.hour

In [29]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_min,date,month,month_name,month_number,day_of_week,hour
0,FBA120ADE8D22E7D,electric_bike,2025-12-26 10:20:09.188,2025-12-26 10:40:13.508,Lafayette Park,JC078,Vesey St & Greenwich St,5216.07,40.713464,-74.062859,40.712547,-74.011131,casual,20.072000,2025-12-26,2025-12,December,12,Friday,10
1,DB8F52B8AE51DD28,electric_bike,2025-12-16 06:48:28.111,2025-12-16 06:56:10.969,Dr. Lena Edwards Park,JC117,Exchange Pl,JC116,40.713203,-74.058261,40.716366,-74.034344,member,7.714300,2025-12-16,2025-12,December,12,Tuesday,6
2,C16D1CA1746E065F,electric_bike,2025-12-20 12:50:55.361,2025-12-20 12:58:32.234,Dr. Lena Edwards Park,JC117,Exchange Pl,JC116,40.713203,-74.058261,40.716366,-74.034344,member,7.614550,2025-12-20,2025-12,December,12,Saturday,12
3,7A613AF4BD6D75B9,electric_bike,2025-12-22 16:59:42.922,2025-12-22 17:04:29.327,JC Medical Center,JC110,Exchange Pl,JC116,40.715391,-74.049692,40.716366,-74.034344,casual,4.773417,2025-12-22,2025-12,December,12,Monday,16
4,2448F2B5B85E52B1,electric_bike,2025-12-17 18:06:09.352,2025-12-17 18:14:13.548,Baldwin at Montgomery,JC020,Exchange Pl,JC116,40.723659,-74.064194,40.716366,-74.034344,member,8.069933,2025-12-17,2025-12,December,12,Wednesday,18


In [30]:
def assign_season(month_number):
    if month_number in [12,1,2]:
        return 'Winter'
    elif month_number in [3,4,5]:
        return 'Spring'
    elif month_number in [6,7,8]:
        return 'Summer'
    else:
        return 'Autumn'

In [31]:
citibike_df['season'] = citibike_df['month_number'].apply(assign_season)

citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,end_lng,member_casual,ride_duration_min,date,month,month_name,month_number,day_of_week,hour,season
0,FBA120ADE8D22E7D,electric_bike,2025-12-26 10:20:09.188,2025-12-26 10:40:13.508,Lafayette Park,JC078,Vesey St & Greenwich St,5216.07,40.713464,-74.062859,...,-74.011131,casual,20.072000,2025-12-26,2025-12,December,12,Friday,10,Winter
1,DB8F52B8AE51DD28,electric_bike,2025-12-16 06:48:28.111,2025-12-16 06:56:10.969,Dr. Lena Edwards Park,JC117,Exchange Pl,JC116,40.713203,-74.058261,...,-74.034344,member,7.714300,2025-12-16,2025-12,December,12,Tuesday,6,Winter
2,C16D1CA1746E065F,electric_bike,2025-12-20 12:50:55.361,2025-12-20 12:58:32.234,Dr. Lena Edwards Park,JC117,Exchange Pl,JC116,40.713203,-74.058261,...,-74.034344,member,7.614550,2025-12-20,2025-12,December,12,Saturday,12,Winter
3,7A613AF4BD6D75B9,electric_bike,2025-12-22 16:59:42.922,2025-12-22 17:04:29.327,JC Medical Center,JC110,Exchange Pl,JC116,40.715391,-74.049692,...,-74.034344,casual,4.773417,2025-12-22,2025-12,December,12,Monday,16,Winter
4,2448F2B5B85E52B1,electric_bike,2025-12-17 18:06:09.352,2025-12-17 18:14:13.548,Baldwin at Montgomery,JC020,Exchange Pl,JC116,40.723659,-74.064194,...,-74.034344,member,8.069933,2025-12-17,2025-12,December,12,Wednesday,18,Winter


#### Storing the enriched data

In [34]:
citibike_df.to_csv(f'{OUTPUT_DIR}/JC/JC2025_Enriched.csv',index=False)